In [1]:
import pandas as pd
import geopandas as gpd


In [2]:
censo_shp = gpd.read_file("../raw_data/dados_censo/SP_setores_CD2022/SP_setores_CD2022.shp")


In [4]:
pd.set_option("display.max_columns", None)

# Importando e tratando os dados considerando a cidade de São Paulo
# Henrique: considerações que eu fiz no processamento
#   - Se a região não tem classificação (i.e., NaN), ipvs é setado como 0
#   - Estou excluindo as estações da CETESB que estão na RMSP, mas não estão na cidade de São Paulo (Osasco, Guarulhos, etc)

# 1. Processamento do IPVS

ipvs = gpd.read_file("../raw_data/ipvs_2022/IPVS_2022.shp")
print(ipvs.columns.tolist())
print(ipvs.head(3))

# Seleção de valores e drops de colunas
ipvs = ipvs[(ipvs["NM_MUN"] == "São Paulo")] # Filtra apenas os setores da cidade de São Paulo
ipvs.loc[ipvs["C_IPVS"].isna(), "C_IPVS"] = 0 # Substitui os valores NaN por 0 na coluna C_IPVS
ipvs.drop(columns=["fid", "CD_MUN", "NM_MUN"], inplace=True)
ipvs = ipvs.reset_index(drop=True)

# O sistema de coordenadas do shapefile é diferente do sistema de coordenadas geográficas (latitude e longitude)
# Então precisamos converter as coordenadas para o sistema geográfico para obter as latitudes e longitudes corretas.

# epsg = numero que o claude falou pra usar pra sp EPSG:31983 (SIRGAS 2000 / UTM zone 23S), 
ipvs_proj = ipvs.to_crs(epsg=31983)
# espg 31983 é de metros, nos queremos em latitude e longitude
centroides = ipvs_proj.geometry.centroid.to_crs(epsg=4326)

# Adicionar as colunas de latitude e longitude ao DataFrame
ipvs["latitude"] = centroides.y # Y = Latitude
ipvs["longitude"] = centroides.x # X = Longitude

print(ipvs.shape)       # quantas linhas e colunas tem
print(ipvs.head())      # primeiras 5 linhas


['fid', 'CD_SETOR', 'SITUACAO', 'CD_MUN', 'NM_MUN', 'CD_DIST', 'NM_DIST', 'N_IPVS', 'C_IPVS', 'geometry']
   fid         CD_SETOR SITUACAO   CD_MUN     NM_MUN    CD_DIST      NM_DIST  \
0  1.0  355030811000549   Urbana  3550308  São Paulo  355030811  Brasilândia   
1  2.0  355030811000548   Urbana  3550308  São Paulo  355030811  Brasilândia   
2  3.0  354160405000053   Urbana  3541604  Promissão  354160405    Promissão   

                  N_IPVS  C_IPVS  \
0  Média Vulnerabilidade     4.0   
1  Média Vulnerabilidade     4.0   
2       Não classificado     NaN   

                                            geometry  
0  POLYGON ((-46.69624 -23.456, -46.69629 -23.456...  
1  POLYGON ((-46.69781 -23.45723, -46.69775 -23.4...  
2  POLYGON ((-49.7252 -21.40299, -49.72527 -21.40...  
(27301, 9)
          CD_SETOR SITUACAO    CD_DIST      NM_DIST                 N_IPVS  \
0  355030811000549   Urbana  355030811  Brasilândia  Média Vulnerabilidade   
1  355030811000548   Urbana  355030811  B

In [5]:
print(censo.head())# 1.5 Adicionar dados de população ao dados do ipvs (IBGE CENSO 2022)
censo = pd.read_csv("../raw_data/dados_censo/Agregados_por_setores_basico_BR_20250417.csv", sep="\t", dtype={"CD_SETOR": str})

# Filtrar só São Paulo (CD_MUN == 355030)
censo_sp = censo[censo["CD_MUN"] == "355030"][["CD_SETOR", "v0001"]]
censo_sp = censo_sp.rename(columns={"v0001": "pop"})

# Converter pop pra numérico (pode ter vírgula como decimal)
censo_sp["pop"] = pd.to_numeric(censo_sp["pop"].str.replace(",", "."), errors="coerce")

# Merge com o IPVS
ipvs = ipvs.merge(censo_sp, on="CD_SETOR", how="left")

NameError: name 'censo' is not defined

In [6]:
# 1.5 Adicionar dados de população ao dados do ipvs (IBGE CENSO 2022)
censo = pd.read_csv(
    "../raw_data/dados_censo/Agregados_por_setores_basico_BR_20250417.csv",
    sep=";",
    dtype={"CD_SETOR": str},
    encoding="latin-1",
    low_memory=False
)
print(censo.columns.tolist())
print(censo.shape)

# Filtrar só São Paulo (CD_MUN == 355030)
censo_sp = censo[censo["CD_MUN"] == 355030][["CD_SETOR", "v0001"]]

# Converter pop pra numérico (pode ter vírgula como decimal)
censo_sp["pop"] = pd.to_numeric(censo_sp["v0001"], errors="coerce")
# Merge com o IPVS
ipvs = ipvs.merge(censo_sp, on="CD_SETOR", how="left")
print(ipvs.head())

['CD_SETOR', 'SITUACAO', 'CD_SIT', 'CD_TIPO', 'AREA_KM2', 'CD_REGIAO', 'NM_REGIAO', 'CD_UF', 'NM_UF', 'CD_MUN', 'NM_MUN', 'CD_DIST', 'NM_DIST', 'CD_SUBDIST', 'NM_SUBDIST', 'CD_BAIRRO', 'NM_BAIRRO', 'CD_NU', 'NM_NU', 'CD_FCU', 'NM_FCU', 'CD_AGLOM', 'NM_AGLOM', 'CD_RGINT', 'NM_RGINT', 'CD_RGI', 'NM_RGI', 'CD_CONCURB', 'NM_CONCURB', 'v0001', 'v0002', 'v0003', 'v0004', 'v0005', 'v0006', 'v0007']
(468099, 36)
          CD_SETOR SITUACAO    CD_DIST      NM_DIST                 N_IPVS  \
0  355030811000549   Urbana  355030811  Brasilândia  Média Vulnerabilidade   
1  355030811000548   Urbana  355030811  Brasilândia  Média Vulnerabilidade   
2  355030811000558   Urbana  355030811  Brasilândia   Alta Vulnerabilidade   
3  355030811000557   Urbana  355030811  Brasilândia   Alta Vulnerabilidade   
4  355030811000556   Urbana  355030811  Brasilândia   Alta Vulnerabilidade   

   C_IPVS                                           geometry   latitude  \
0     4.0  POLYGON ((-46.69624 -23.456, -46.6962

In [ ]:
print(ipvs[["CD_SETOR", "C_IPVS", "pop"]].head(10))
print("\nNulos em pop:", ipvs["pop"].isna().sum())
print("Total de setores:", len(ipvs))

In [ ]:
print("IPVS:", ipvs["CD_SETOR"].iloc[0], type(ipvs["CD_SETOR"].iloc[0]))
print("Censo:", censo["CD_SETOR"].iloc[0], type(censo["CD_SETOR"].iloc[0]))

In [ ]:
print(censo["CD_MUN"].dtype)
print(censo["CD_MUN"].iloc[0], type(censo["CD_MUN"].iloc[0]))
censo_sp = censo[censo["CD_MUN"] == "355030"][["CD_SETOR", "v0001"]]

In [12]:
censo = pd.read_csv(
    "../raw_data/dados_censo/Agregados_por_setores_basico_BR_20250417.csv",
    sep=";",
    dtype={"CD_SETOR": str, "CD_MUN": str},  # força CD_MUN como string também
    encoding="latin-1",
    low_memory=False
)

censo_sp = censo[censo["CD_MUN"] == "3550308"][["CD_SETOR", "v0001"]]
censo_sp = censo_sp.rename(columns={"v0001": "pop"})
censo_sp["pop"] = pd.to_numeric(censo_sp["pop"], errors="coerce")

ipvs = ipvs.merge(censo_sp, on="CD_SETOR", how="left")

print(ipvs[["CD_SETOR", "C_IPVS", "pop"]].head(10))
print(censo_sp.shape)
print(censo_sp.head())
print("\nNulos em pop:", ipvs["pop"].isna().sum())

          CD_SETOR  C_IPVS  pop
0  355030811000549     4.0  447
1  355030811000548     4.0  418
2  355030811000558     5.0  501
3  355030811000557     5.0  315
4  355030811000556     5.0  455
5  355030811000555     4.0  372
6  355030811000554     5.0  385
7  355030811000552     2.0  287
8  355030811000559     5.0  430
9  355030811000561     5.0  405
(27301, 2)
               CD_SETOR   pop
331734  355030801000001   682
331735  355030801000002  1374
331736  355030801000003   557
331737  355030801000004   526
331738  355030801000005   579

Nulos em pop: 0


In [9]:
print(censo_sp.shape)
print(censo_sp.head())
print(censo["CD_MUN"].unique()[:10])

(0, 2)
Empty DataFrame
Columns: [CD_SETOR, pop]
Index: []
<StringArray>
['1100015', '1100023', '1100031', '1100049', '1100056', '1100064', '1100072',
 '1100080', '1100098', '1100106']
Length: 10, dtype: str
